# E-Commerce Sales Insights
## Exploratory Data Analysis & KPI Dashboard
**Project by:** Shantanu Shrivastava

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded successfully!')

## 1. Data Loading & Initial Exploration

In [ ]:
df = pd.read_csv('../data/ecommerce_transactions.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

In [ ]:
print('Data Types:')
print(df.dtypes)
print('\nDescriptive Statistics:')
df.describe()

In [ ]:
print('Missing Values:')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})

## 2. Data Cleaning & Preprocessing

In [ ]:
# Convert date column
df['order_date'] = pd.to_datetime(df['order_date'])

# Extract time features
df['year']    = df['order_date'].dt.year
df['month']   = df['order_date'].dt.month
df['month_name'] = df['order_date'].dt.strftime('%b')
df['quarter'] = df['order_date'].dt.quarter
df['day_of_week'] = df['order_date'].dt.day_name()

# Fill missing values
df['city'].fillna('Unknown', inplace=True)
df['payment_method'].fillna('Unknown', inplace=True)

# Standardize category names
df['product_category'] = df['product_category'].str.strip().str.title()

print('Data cleaned successfully!')
print(f'Date range: {df["order_date"].min().date()} to {df["order_date"].max().date()}')
df.head()

## 3. Key Performance Indicators (KPIs)

In [ ]:
total_sales      = df['total_price'].sum()
total_orders     = df['order_id'].nunique()
avg_order_value  = total_sales / total_orders
total_customers  = df['customer_id'].nunique()
total_units_sold = df['quantity'].sum()
best_category    = df.groupby('product_category')['total_price'].sum().idxmax()

print('=' * 50)
print('       KEY PERFORMANCE INDICATORS')
print('=' * 50)
print(f'  Total Revenue       : Rs {total_sales:,.2f}')
print(f'  Total Orders        : {total_orders:,}')
print(f'  Avg Order Value     : Rs {avg_order_value:,.2f}')
print(f'  Unique Customers    : {total_customers:,}')
print(f'  Total Units Sold    : {total_units_sold:,}')
print(f'  Best Category       : {best_category}')
print('=' * 50)

In [ ]:
# KPI Cards visualization
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
fig.patch.set_facecolor('#f8f9fa')

kpis = [
    ('Total Revenue', f'Rs {total_sales/1e6:.1f}M', '#2ecc71'),
    ('Total Orders', f'{total_orders:,}', '#3498db'),
    ('Avg Order Value', f'Rs {avg_order_value:,.0f}', '#9b59b6'),
    ('Unique Customers', f'{total_customers:,}', '#e67e22'),
]

for ax, (title, value, color) in zip(axes, kpis):
    ax.set_facecolor(color)
    ax.text(0.5, 0.6, value, transform=ax.transAxes,
            fontsize=22, fontweight='bold', color='white',
            ha='center', va='center')
    ax.text(0.5, 0.2, title, transform=ax.transAxes,
            fontsize=11, color='white', ha='center', va='center')
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

plt.suptitle('E-Commerce KPI Dashboard', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/kpi_cards.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Sales Trend Analysis

In [ ]:
# Monthly sales trend
monthly_sales = df.groupby(['year', 'month'])['total_price'].sum().reset_index()
monthly_sales['period'] = pd.to_datetime(monthly_sales[['year','month']].assign(day=1))
monthly_sales = monthly_sales.sort_values('period')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Line chart - monthly trend
ax1.plot(monthly_sales['period'], monthly_sales['total_price'],
         color='#3498db', linewidth=2.5, marker='o', markersize=5)
ax1.fill_between(monthly_sales['period'], monthly_sales['total_price'],
                 alpha=0.15, color='#3498db')
ax1.set_title('Monthly Sales Trend', fontsize=13, fontweight='bold')
ax1.set_xlabel('Month')
ax1.set_ylabel('Total Sales (Rs)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs {x/1e3:.0f}K'))
ax1.tick_params(axis='x', rotation=45)

# Bar chart - quarterly sales
quarterly = df.groupby(['year','quarter'])['total_price'].sum().reset_index()
quarterly['label'] = quarterly.apply(lambda r: f"Q{r['quarter']}\n{r['year']}", axis=1)
colors = ['#2ecc71','#3498db','#9b59b6','#e67e22'] * 2
ax2.bar(quarterly['label'], quarterly['total_price'],
        color=colors[:len(quarterly)], edgecolor='white', linewidth=0.5)
ax2.set_title('Quarterly Sales Comparison', fontsize=13, fontweight='bold')
ax2.set_ylabel('Total Sales (Rs)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs {x/1e3:.0f}K'))

plt.tight_layout()
plt.savefig('../reports/sales_trend.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Category & Product Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sales by category
cat_sales = df.groupby('product_category')['total_price'].sum().sort_values(ascending=True)
cat_sales.plot(kind='barh', ax=axes[0], color='#3498db', edgecolor='white')
axes[0].set_title('Sales by Category', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Total Sales (Rs)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

# Orders by category
cat_orders = df.groupby('product_category')['order_id'].count().sort_values(ascending=False)
colors_pie = ['#2ecc71','#3498db','#9b59b6','#e67e22','#e74c3c','#1abc9c']
axes[1].pie(cat_orders, labels=cat_orders.index, autopct='%1.1f%%',
            colors=colors_pie, startangle=90,
            wedgeprops={'edgecolor':'white', 'linewidth':1.5})
axes[1].set_title('Order Distribution by Category', fontsize=13, fontweight='bold')

# Top 10 products by revenue
top_products = df.groupby('product_name')['total_price'].sum().sort_values(ascending=False).head(10)
top_products.plot(kind='bar', ax=axes[2], color='#9b59b6', edgecolor='white')
axes[2].set_title('Top 10 Products by Revenue', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Total Sales (Rs)')
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../reports/category_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Customer Behavior Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Payment method distribution
payment = df['payment_method'].value_counts()
payment.plot(kind='bar', ax=axes[0], color=['#3498db','#2ecc71','#e67e22','#9b59b6','#e74c3c'],
             edgecolor='white')
axes[0].set_title('Payment Method Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Orders')
axes[0].tick_params(axis='x', rotation=30)

# Top 10 cities by sales
city_sales = df[df['city'] != 'Unknown'].groupby('city')['total_price'].sum().sort_values(ascending=True).tail(10)
city_sales.plot(kind='barh', ax=axes[1], color='#1abc9c', edgecolor='white')
axes[1].set_title('Top Cities by Revenue', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Total Sales (Rs)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

# Order quantity distribution
qty_dist = df['quantity'].value_counts().sort_index()
axes[2].bar(qty_dist.index, qty_dist.values,
            color=['#3498db','#2ecc71','#e67e22','#9b59b6','#e74c3c'],
            edgecolor='white')
axes[2].set_title('Order Quantity Distribution', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Quantity per Order')
axes[2].set_ylabel('Number of Orders')

plt.tight_layout()
plt.savefig('../reports/customer_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
numeric_cols = df[['quantity', 'unit_price', 'total_price', 'month', 'quarter']]
corr = numeric_cols.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            ax=ax, linewidths=0.5, square=True,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

## 8. Summary & Key Insights

In [ ]:
best_month_data = monthly_sales.loc[monthly_sales['total_price'].idxmax()]
best_month = best_month_data['period'].strftime('%B %Y')
best_city = df[df['city'] != 'Unknown'].groupby('city')['total_price'].sum().idxmax()
top_payment = df['payment_method'].value_counts().index[0]

print('KEY INSIGHTS')
print('=' * 55)
print(f'  1. Total Revenue generated: Rs {total_sales:,.2f}')
print(f'  2. Best performing month   : {best_month}')
print(f'  3. Top product category    : {best_category}')
print(f'  4. Most preferred payment  : {top_payment}')
print(f'  5. Top city by revenue     : {best_city}')
print(f'  6. Avg order value         : Rs {avg_order_value:,.2f}')
print(f'  7. Repeat customers likely : {total_orders - total_customers} repeat orders')
print('=' * 55)
print('\nAll charts saved in /reports/ folder!')